# Quantise after training (post-training quantisation)

Train a neural ranking function in the **reals**, then quantise it to integer weights and
verify the certificate in **LIA**. The lesson: naive rounding collapses the certificate —
you need fixed-point **scaling**, and the margin scales with it.

The program, training loop, `quantise` step, and LIA `verify` live in `_ranking.py` (usage
code, not part of the framework).

In [ ]:
import torch
import sys
from pathlib import Path

# find the sibling helper module whether we run from the repo root or tutorials/
for _d in (Path.cwd(), Path.cwd() / "tutorials"):
    if (_d / "_ranking.py").exists():
        sys.path.insert(0, str(_d)); break
import _ranking as rk

## The integer program

An integer counter (`sugar.Module`, theory `LIA`): `x` increments while `x < y` or `x < z`,
and resets otherwise; `y`, `z` are constant targets. We certify that it always reaches a target.

In [ ]:
program, state_vars = rk.make_program()
print(program.with_varnames(dict(zip(state_vars, "xyz"))))

module
  interface
    x : Int([1,1])
    y : Int([1,1])
    z : Int([1,1])
  atom controls x, y, z reads x, y, z
  init
    #9 := [[0]] 
    X(x) := Id #9
    #10 := [[0]] 
    X(y) := Id #10
    #11 := [[0]] 
    X(z) := Id #11
  delay
    d(x) := ZERO 
    d(y) := ZERO 
    d(z) := ZERO 
  update
    #12 := Lt (x, y)
    #13 := Lt (x, z)
    #14 := Or (#12, #13)
    #15 := [[1]] 
    #16 := Add (x, #15)
    #17 := [[0]] 
    #18 := Ite (#14, #16, #17)
    X(x) := Id #18
    X(y) := Id y
    X(z) := Id z



## Train the ranking function in the reals

`R(x, y, z)` is wrapped as a `zrth.torch.Module` (theory `LRA`) and trained so that, away from
a target, `R >= 0` and `R` decreases by at least `margin` each step.

In [ ]:
torch.manual_seed(0)
real_V = rk.Module(rk.RankingNN(), combinatorial=True)     # LRA, float — trainable
rk.train(real_V, margin=1.0)
print("trained (real / LRA)")

trained (real / LRA)


## Quantise after training — naive rounding degenerates

`quantise(real_V, scale=1)` just rounds the weights to integers. Trained weights are small
(`|w| < 1`), so most round to **0**: the certificate collapses and the *decrease* condition fails.

In [ ]:
naive_V = rk.quantise(real_V, scale=1)
print("scale=1  verify (V>=0, decreases):", rk.verify(program, naive_V, state_vars, delta=1))

scale=1  verify (V>=0, decreases): (True, False)


## Fixed-point scaling

Scale before rounding — `round(scale * w)` — so the weights survive, and scale the margin too
(`delta' = scale * delta`; both `V >= 0` and `V(s) - V(s') >= delta` are scale-invariant). At
`scale = 4` the integer certificate verifies.

In [ ]:
scale = 4
int_V = rk.quantise(real_V, scale)          # the "same module, but int"
print(f"scale={scale}  verify (V>=0, decreases):",
      rk.verify(program, int_V, state_vars, delta=1 * scale))

scale=4  verify (V>=0, decreases): (True, True)


## Result

The real ranking function, quantised with fixed-point scaling, verifies as an integer (LIA)
certificate. Post-training quantisation is simple but *post-hoc*: rounding can break the
just-trained margin (the `scale=1` case), so it needs a large enough scale. The companion
tutorial trains *through* the rounding (quantisation-aware) so the integer certificate holds
by construction.